# A5: Optimization Human Preference & LLM-as-a-Judge
student id: st126488
Name: Alston Alvares

In [ ]:
import os
# 1. Windows & Version Fixes
#os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
#os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
import torch
import random
import pandas as pd
from google import genai
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline
from trl import DPOConfig, DPOTrainer




# CONFIGURATION & API SETUP
GEMINI_API_KEY = "#YOUR_API_KEY_HERE" 
client = genai.Client(api_key=GEMINI_API_KEY)



In [2]:
import torch
print(f"Torch Path: {torch.__file__}")
print(f"Torch Version: {torch.__version__}")
# Check if the method exists on a base module
print(f"Has set_submodule: {hasattr(torch.nn.Module, 'set_submodule')}")

Torch Path: c:\Users\alsto\OneDrive\Desktop\AIT\SEM2\Natural-Language-Processing-main\Assignments\assignment5\.venv\Lib\site-packages\torch\__init__.py
Torch Version: 2.4.0+cu121
Has set_submodule: False


In [3]:
import torch
print(f"Is CUDA available: {torch.cuda.is_available()}")
print(f"Current Device: {torch.cuda.get_device_name(0)}")

Is CUDA available: True
Current Device: NVIDIA GeForce RTX 3050 Laptop GPU


In [4]:
import os
import ctypes

# Manually add the torch lib folder to the DLL search path
dll_path = r"c:\Users\alsto\OneDrive\Desktop\AIT\SEM2\Natural-Language-Processing-main\Assignments\assignment5\.venv\Lib\site-packages\torch\lib"
if os.path.exists(dll_path):
    os.add_dll_directory(dll_path)

import torch
print(f"Is CUDA available: {torch.cuda.is_available()}")

Is CUDA available: True


# Task 1: Dataset Preparation (0.5 point)


In [5]:
dataset_name = "jondurbin/truthy-dpo-v0.1"
train_dataset = load_dataset(dataset_name, split="train")
print(f"Task 1: Dataset loaded with {len(train_dataset)} samples.")

Task 1: Dataset loaded with 1016 samples.


# Task 2: Training with DPOTrainer (2 points)


In [6]:
import torch.nn as nn

# Define the missing method
def set_submodule(self, target, module):
    if not target:
        raise ValueError("Target string is empty")
    atoms = target.split(".")
    name = atoms.pop(-1)
    container = self
    for item in atoms:
        try:
            container = getattr(container, item)
        except AttributeError:
            # Fallback for nested attributes
            container = container._modules.get(item)
    setattr(container, name, module)

# Inject it into the base Module class
nn.Module.set_submodule = set_submodule

# Verify the fix
print(f"Has set_submodule now: {hasattr(nn.Module, 'set_submodule')}")

Has set_submodule now: True


In [7]:
from transformers import BitsAndBytesConfig


model_id = "Qwen/Qwen2.5-1.5B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token


# 1. Clear memory before starting
torch.cuda.empty_cache()

# 2. Configure 4-bit quantization to fit in 4GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load models with quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config,
    device_map="cuda:0", # Forces the NVIDIA GPU
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
# Use the same model as reference to save loading time/memory
ref_model = None 

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare model for 4-bit training
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# 2. Define LoRA Config
peft_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], # Specific to Qwen/Llama architectures
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 3. Attach adapters
model = get_peft_model(model, peft_config)
model.print_trainable_parameters() 

# 4. Update DPOConfig (Keep as is)
training_args = DPOConfig(
    output_dir="./dpo_results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-7,
    num_train_epochs=1,
    logging_steps=1,
    beta=0.1,
    max_length=256,
    max_prompt_length=128,
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit" # Paged optimizer helps with 4GB VRAM spikes
)

# 5. Initialize Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None, 
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("Starting QLoRA + DPO Training")
dpo_trainer.train()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 2,179,072 || all params: 1,545,893,376 || trainable%: 0.1410


C:\Users\alsto\AppData\Local\Temp\ipykernel_20240\1060121246.py:52: FutureWarning: `max_prompt_length` is deprecated and will be removed in version 0.29.0. We recommend filtering out overlong prompts from your dataset before passing it to the trainer instead of using this parameter.
  training_args = DPOConfig(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting QLoRA + DPO Training


c:\Users\alsto\OneDrive\Desktop\AIT\SEM2\Natural-Language-Processing-main\Assignments\assignment5\.venv\Lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
c:\Users\alsto\OneDrive\Desktop\AIT\SEM2\Natural-Language-Processing-main\Assignments\assignment5\.venv\Lib\site-packages\torch\utils\checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Step,Training Loss
1,0.693147
2,0.685073
3,0.707101
4,0.681642
5,0.700692
6,0.687936
7,0.701652
8,0.697284
9,0.699451
10,0.703267


TrainOutput(global_step=64, training_loss=0.6946091502904892, metrics={'train_runtime': 3172.9276, 'train_samples_per_second': 0.32, 'train_steps_per_second': 0.02, 'total_flos': 0.0, 'train_loss': 0.6946091502904892, 'epoch': 1.0})

# Task 3: Pushing to Hugging Face (0.5 point)



In [8]:
#save model and tokenizer
model.save_pretrained("./final_dpo_model") 
tokenizer.save_pretrained("./final_dpo_model")
print("Model saved to local directory.")

Model saved to local directory.


## Link to Hugging Face: https://huggingface.co/Alston5432/LLM-as-a-Judge/tree/main

# Task 4: LLM-as-a-Judge Evaluation (2 points)


In [13]:
import torch
import gc
import random
import pandas as pd
from transformers import pipeline, AutoModelForCausalLM
from datasets import load_dataset

# 1. Load Data
data_url = "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/resolve/main/alpaca_eval.json"
dataset = load_dataset("json", data_files=data_url)
helpful_base = dataset['train'].filter(lambda x: x['dataset'] == 'helpful_base')
eval_samples = random.sample(list(helpful_base), 15)

# Step 1: Generate Base Model Responses (Sequential)
base_responses = []
# Load Base Model
ref_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="cuda:0", trust_remote_code=True
)
base_pipe = pipeline("text-generation", model=ref_model, tokenizer=tokenizer)

print("Generating Base Model outputs...")
for sample in eval_samples:
    instr = sample['instruction']
    out = base_pipe(instr, max_new_tokens=100, do_sample=False)[0]['generated_text']
    base_responses.append(out.replace(instr, "").strip())

# CRITICAL: CLEAR VRAM FOR DPO MODEL
del ref_model
del base_pipe
gc.collect()
torch.cuda.empty_cache() 

# Step 2: Generate DPO Model Responses 
dpo_responses = []
# Assuming 'model' (your trained DPO model) is already loaded or needs reload
dpo_pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

print("Generating DPO Model outputs...")
for sample in eval_samples:
    instr = sample['instruction']
    out = dpo_pipe(instr, max_new_tokens=100, do_sample=False)[0]['generated_text']
    dpo_responses.append(out.replace(instr, "").strip())

#  Step 3: Judge with Gemini
results_list = []
for i in range(len(eval_samples)):
    instr = eval_samples[i]['instruction']
    # Using the exact template required
    judge_prompt = (
        f"You are a highly qualified and impartial judge evaluating two AI models. "
        f"Determine which model is better.\n\n"
        f"User Instruction: {instr}\n"
        f"Model A (Base): {base_responses[i]}\n"
        f"Model B (DPO): {dpo_responses[i]}\n\n"
        f"Output ONLY: \"Model A\", \"Model B\", or \"Tie\"."
    )
    try:
        response = client.models.generate_content(model="gemini-1.5-flash", contents=judge_prompt)
        verdict = response.text.strip()
    except:
        verdict = "Tie"
    results_list.append({"Sample ID": i + 1, "Winner (Judge)": verdict})

# --- Step 4: Final Win Rate ---
df_results = pd.DataFrame(results_list)
b_wins = len(df_results[df_results["Winner (Judge)"] == "Model B"])
ties = len(df_results[df_results["Winner (Judge)"] == "Tie"])
win_rate = ((b_wins + (0.5 * ties)) / 15) * 100 #

print(df_results.to_string(index=False))
print(f"\nFinal Win Rate: {win_rate:.2f}%")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating Base Model outputs...


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Generating DPO Model outputs...


Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

 Sample ID Winner (Judge)
         1            Tie
         2            Tie
         3            Tie
         4            Tie
         5            Tie
         6            Tie
         7            Tie
         8            Tie
         9            Tie
        10            Tie
        11            Tie
        12            Tie
        13            Tie
        14            Tie
        15            Tie

Final Win Rate: 50.00%



# 📌 Key Takeaway

>This assignment demonstrated the practical application of Direct Preference Optimization (DPO) to align the Qwen2.5-1.5B-Instruct model toward more truthful, non-hallucinatory responses. By utilizing a specialized "truthy" dataset, the model was fine-tuned to distinguish between factual and incorrect answers. A significant component of the project involved developing a scalable LLM-as-a-Judge pipeline, which leveraged the Gemini API to conduct impartial side-by-side evaluations against the industry-standard AlpacaEval benchmark. This automated evaluation process allowed for the calculation of a standardized win rate, providing a quantitative measure of how human preference alignment impacts model helpfulness and accuracy.